# Async HITL via queue

Long-running agents can't hold an HTTP request open while waiting for a human. The fix is the classic event-driven pattern:

1. Agent runs until `interrupt()`.
2. Server persists `{thread_id, payload}` onto a queue and returns 202.
3. A reviewer pulls the queue (Slack message, dashboard, email) and POSTs back `{thread_id, decision}`.
4. A webhook handler calls `graph.resume(thread_id, Command(resume=decision))` and the agent continues.

In [ ]:
import os, sys, pathlib
from collections import deque
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / '04-human-in-the-loop'))
os.environ.setdefault('LLM_CACHE_ONLY', '1')

from hitl import build_research_graph, Checkpointer, Command, get_question

graph = build_research_graph(); cp = Checkpointer()
queue = deque()

for qid in ('q01', 'q23'):
    state = graph.run({'question': get_question(qid), 'question_id': qid},
                      thread_id=qid, checkpointer=cp)
    queue.append({'thread_id': qid, 'draft': state['draft']})
list(queue)

In [ ]:
# Reviewer drains the queue and posts a decision per thread.
while queue:
    req = queue.popleft()
    decision = {'approved': True, 'reviewer': f'queue:{req["thread_id"]}'}
    state = graph.resume(thread_id=req['thread_id'],
                         command=Command(resume=decision),
                         checkpointer=cp)
    print(req['thread_id'], '->', state['final_answer'][:80])

## Production wiring (FastAPI sketch)

```python
@app.post('/agents/{thread_id}/decide')
async def decide(thread_id: str, decision: dict):
    state = graph.resume(thread_id=thread_id,
                         command=Command(resume=decision),
                         checkpointer=PostgresSaver(pool))
    return {'published': state['published']}
```

Swap `deque` for SQS / Redis / a Postgres table, and `PostgresSaver` for the real LangGraph checkpointer — the agent code is unchanged.